# 🚀 Bhusawal Connect — Google Colab Background Worker Engine

This Google Colab notebook acts as the high-performance processing worker for the **Antigravity ↔ Google Sheets / Google Drive ↔ Google Colab** integration.

### ⚡ What this worker does:
1. **Mounts Google Drive** and authenticates with your private Google Sheets.
2. **Polls the `_Jobs_Queue` tab** for tasks created by Antigravity (e.g. *"Process my product catalogue and generate accurate images"*).
3. **Processes the 7 required catalogue fields**:
   - `Product Name`
   - `Brand`
   - `Quantity`
   - `MRP` (Used as variant signal; **NEVER** rendered on images)
   - `Image URL`
   - `Image File` (Deterministic: `product-name_brand_quantity_mrp.jpg`)
   - `Image Verification Status` (`VERIFIED`, `PENDING_GENERATION`, `REJECTED`)
4. **Generates Zepto/Instamart-style clean commercial product photography** on pure solid white backgrounds.
5. **Saves images to Google Drive** and updates the Google Sheet in real-time.
6. **Streams live progress (%) and logs** to the `_Job_Logs` tab.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install --upgrade gspread google-auth Pillow requests

## 🔐 Step 2: Authenticate with Google & Mount Google Drive

In [ ]:
from google.colab import auth, drive
import os

# 1. Authenticate user session
print("🔐 Authenticating with Google...")
auth.authenticate_user()

# 2. Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

## ⚙️ Step 3: Configure Worker Parameters
Your target Sheet ID is pre-filled below:

In [ ]:
# Set your Google Sheet ID (pre-filled with your active catalogue sheet)
GOOGLE_SHEET_ID = "1CdU03pY_tAUTXND4NH8KI2siE6gcR2XqOPgBkaQCHVY"  # @param {type:"string"}
GOOGLE_SHEET_NAME = "Bhusawal_Connect_Master_Catalogue"  # @param {type:"string"}
DRIVE_IMAGE_FOLDER = "/content/drive/MyDrive/BhusawalConnect/ProductImages"  # @param {type:"string"}

os.makedirs(DRIVE_IMAGE_FOLDER, exist_ok=True)
print(f"📁 Output Folder set to: {DRIVE_IMAGE_FOLDER}")
print(f"📊 Target Sheet ID: {GOOGLE_SHEET_ID}")

## 🧠 Step 4: Load Colab Worker Engine

In [ ]:
import sys
import time
import json
import re
import random
import urllib.parse
import urllib.request
from datetime import datetime
from pathlib import Path
import gspread
from google.auth import default

def sanitize_filename(text):
    cleaned = re.sub(r'[^a-zA-Z0-9_-]', '_', str(text).strip().lower())
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned[:80]

class BhusawalColabWorker:
    def __init__(self, sheet_id=None, sheet_name="Bhusawal_Connect_Master_Catalogue", drive_folder_path="/content/drive/MyDrive/BhusawalConnect/ProductImages"):
        self.sheet_id = (sheet_id or "1CdU03pY_tAUTXND4NH8KI2siE6gcR2XqOPgBkaQCHVY").strip()
        self.sheet_name = sheet_name
        self.drive_folder_path = Path(drive_folder_path)
        self.drive_folder_path.mkdir(parents=True, exist_ok=True)
        creds, _ = default()
        self.gc = gspread.authorize(creds)
        self.open_spreadsheet()
        self.ensure_worksheets()

    def open_spreadsheet(self):
        if self.sheet_id:
            self.spreadsheet = self.gc.open_by_key(self.sheet_id)
        else:
            self.spreadsheet = self.gc.open(self.sheet_name)
        print(f"✅ Connected to: '{self.spreadsheet.title}'")

    def ensure_worksheets(self):
        existing = [ws.title for ws in self.spreadsheet.worksheets()]
        if "Product_Catalogue" not in existing:
            ws = self.spreadsheet.add_worksheet(title="Product_Catalogue", rows=1000, cols=10)
            ws.append_row(["Product Name", "Brand", "Quantity", "MRP", "Image URL", "Image File", "Image Verification Status"])
        if "_Jobs_Queue" not in existing:
            ws = self.spreadsheet.add_worksheet(title="_Jobs_Queue", rows=500, cols=16)
            ws.append_row(["Job ID", "Task", "Created At", "Status", "Progress", "Total Items", "Completed Items", "Failed Items", "Pending Items", "Current Product", "Error", "Started At", "Completed At", "Summary"])
        if "_Job_Logs" not in existing:
            ws = self.spreadsheet.add_worksheet(title="_Job_Logs", rows=2000, cols=8)
            ws.append_row(["Timestamp", "Job ID", "Level", "Item / SKU", "Message"])

    def log(self, job_id, level, item_sku, message):
        ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{ts}] [{job_id}] [{level}] ({item_sku}) {message}")
        try:
            ws = self.spreadsheet.worksheet("_Job_Logs")
            ws.append_row([ts, job_id, level, str(item_sku), str(message)])
        except Exception:
            pass

    def update_job_status(self, job_id, row_num, status, progress="0%", total_items=0, completed_items=0, failed_items=0, pending_items=0, current_product="", error_msg="", summary=""):
        try:
            ws = self.spreadsheet.worksheet("_Jobs_Queue")
            now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
            ws.update_cell(row_num, 4, status)
            ws.update_cell(row_num, 5, progress)
            ws.update_cell(row_num, 6, total_items)
            ws.update_cell(row_num, 7, completed_items)
            ws.update_cell(row_num, 8, failed_items)
            ws.update_cell(row_num, 9, pending_items)
            ws.update_cell(row_num, 10, current_product[:40])
            if status == "RUNNING" and not ws.cell(row_num, 12).value:
                ws.update_cell(row_num, 12, now)
            if status in ("COMPLETED", "FAILED", "CANCELLED"):
                ws.update_cell(row_num, 13, now)
            if error_msg:
                ws.update_cell(row_num, 11, error_msg)
            if summary:
                ws.update_cell(row_num, 14, summary)
        except Exception as e:
            print(f"Status update error: {e}")

    def generate_and_verify_image(self, product_name, brand, quantity, mrp):
        clean_name = str(product_name).strip()
        clean_brand = str(brand).strip() or "Generic"
        clean_qty = str(quantity).strip() or "1 unit"
        clean_mrp = str(mrp).replace("₹", "").strip() or "0"

        slug = sanitize_filename(f"{clean_name}_{clean_brand}_{clean_qty}_{clean_mrp}")
        filename = f"{slug}.jpg"
        save_path = self.drive_folder_path / filename

        if save_path.exists() and save_path.stat().st_size > 5000:
            direct_url = f"https://image.pollinations.ai/prompt/{urllib.parse.quote(clean_name)}?width=800&height=800"
            return direct_url, filename, "VERIFIED"

        prompt = (
            f"Centered front-facing studio product photography of authentic {clean_brand} {clean_name}, "
            f"official commercial packaging for {clean_qty} pack size. "
            f"Clean solid seamless pure white background, professional studio softbox lighting, ultra-sharp focus, "
            f"8k resolution, photorealistic grocery e-commerce catalog image. "
            f"NO price tags, NO MRP text, NO currency symbols, NO discount stickers, NO promotional badges, NO watermarks."
        )
        encoded_prompt = urllib.parse.quote(prompt)
        ai_url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=800&height=800&nologo=true&seed={random.randint(1000, 999999)}"

        for attempt in range(3):
            try:
                req = urllib.request.Request(ai_url, headers={"User-Agent": "BhusawalConnectColabWorker/2.0"})
                with urllib.request.urlopen(req, timeout=30) as resp:
                    img_data = resp.read()
                    if len(img_data) > 5000:
                        with open(save_path, "wb") as f:
                            f.write(img_data)
                        return ai_url, filename, "VERIFIED"
            except Exception:
                time.sleep(2 * (attempt + 1))
        return None, None, "REJECTED"

    def execute_job(self, job):
        job_id = job.get("Job ID", "UNKNOWN")
        row_num = job.get("_row_number", 2)
        task = job.get("Task", "PROCESS_CATALOG")
        req_limit = int(job.get("Total Items", 0)) or 50

        print(f"\n========================================")
        print(f"New job detected: {job_id}")
        print(f"Task: {task}")
        print(f"Products: {req_limit}")
        print(f"========================================")

        self.update_job_status(job_id, row_num, "RUNNING", progress="0%", total_items=req_limit, pending_items=req_limit)
        self.log(job_id, "INFO", "CLAIMED", f"Processing catalogue (limit {req_limit})...")

        try:
            ws = self.spreadsheet.worksheet("Product_Catalogue")
            records = ws.get_all_records()
            total_items = min(len(records), req_limit) if records else 0

            if total_items == 0:
                self.update_job_status(job_id, row_num, "COMPLETED", progress="100%", total_items=0, completed_items=0, failed_items=0, pending_items=0, summary="Catalogue is empty (0 products processed).")
                print(f"Job completed: {job_id} (0 products)")
                return

            completed, failed = 0, 0
            for idx, item in enumerate(records[:total_items], start=2):
                name = str(item.get("Product Name", "")).strip()
                brand = str(item.get("Brand", "")).strip()
                qty = str(item.get("Quantity", "1 unit")).strip()
                mrp = str(item.get("MRP", "0")).strip()
                current_status = str(item.get("Image Verification Status", "")).strip().upper()
                current_url = str(item.get("Image URL", "")).strip()

                print(f"Processing {idx - 1}/{total_items}: {name}")

                if not name:
                    completed += 1
                    continue

                needs_processing = (
                    current_status != "VERIFIED" or
                    not current_url or
                    "placeholder" in current_url.lower()
                )

                if needs_processing:
                    self.log(job_id, "INFO", name, f"Generating studio image for {brand} {name}...")
                    img_url, img_file, v_status = self.generate_and_verify_image(name, brand, qty, mrp)
                    if v_status == "VERIFIED" and img_url:
                        ws.update_cell(idx, 5, img_url)
                        ws.update_cell(idx, 6, img_file)
                        ws.update_cell(idx, 7, "VERIFIED")
                        completed += 1
                        self.log(job_id, "SUCCESS", name, f"Verified: {img_file}")
                    else:
                        ws.update_cell(idx, 7, "REJECTED")
                        failed += 1
                        self.log(job_id, "ERROR", name, "Image verification failed. Marked REJECTED.")
                else:
                    completed += 1

                processed = completed + failed
                pending = max(0, total_items - processed)
                pct_str = f"{(processed / total_items) * 100:.0f}%"
                self.update_job_status(job_id, row_num, "RUNNING", progress=pct_str, total_items=total_items, completed_items=completed, failed_items=failed, pending_items=pending, current_product=name)

            summary = f"Processed {total_items} items. Successful: {completed}, Failed: {failed}."
            self.update_job_status(job_id, row_num, "COMPLETED", progress="100%", total_items=total_items, completed_items=completed, failed_items=failed, pending_items=0, current_product="Done", summary=summary)
            self.log(job_id, "INFO", "DONE", f"✅ Completed: {summary}")
            print(f"\n========================================")
            print(f"Job completed: {job_id}")
            print(f"Successful: {completed}")
            print(f"Failed: {failed}")
            print(f"========================================\n")
        except Exception as e:
            self.update_job_status(job_id, row_num, "FAILED", error_msg=str(e))
            self.log(job_id, "CRITICAL", "FAILURE", str(e))

    def run_worker_loop(self):
        print("========================================")
        print("Bhusawal Connect Colab Worker")
        print("========================================")
        print("Authenticated: YES")
        print("Google Drive: CONNECTED")
        print(f"Google Sheets: CONNECTED ({self.spreadsheet.title})")
        print("Worker: RUNNING\n")
        print("Waiting for jobs...")
        while True:
            try:
                ws = self.spreadsheet.worksheet("_Jobs_Queue")
                rows = ws.get_all_records()
                queued = [dict(r, _row_number=idx) for idx, r in enumerate(rows, start=2) if str(r.get("Status", "")).strip().upper() in ("QUEUED", "PENDING")]
                if queued:
                    for job in queued:
                        self.execute_job(job)
                    print("Waiting for jobs...")
                else:
                    time.sleep(5)
            except KeyboardInterrupt:
                print("\nWorker stopped.")
                break
            except Exception as e:
                time.sleep(5)

print("✅ Worker Engine Loaded Successfully!")

## 🚀 Step 5: Start Worker Daemon Loop
Run this cell to start processing jobs automatically!

In [ ]:
worker = BhusawalColabWorker(
    sheet_id=GOOGLE_SHEET_ID,
    sheet_name=GOOGLE_SHEET_NAME,
    drive_folder_path=DRIVE_IMAGE_FOLDER
)

# Start listening for jobs submitted by Antigravity
worker.run_worker_loop()